In [1]:
from pathlib import Path
import sys
import numpy as np

folder = Path.cwd()
candidates = [
    folder / "src",
    folder.parent / "src",
    folder / "sib" / "src"
]

src = next(path for path in candidates if (path / "si").is_dir())
sys.path.insert(0, str(src))

from si.neural_networks.layers import DenseLayer

In [2]:
layer = DenseLayer(4, (3,)).initialize()

out = layer.forward_propagation(np.random.rand(10, 3))

assert out.shape == (10, 4)
assert layer.parameters() == 3 * 4 + 4
assert layer.output_shape() == (4,)

print("Todos os testes passaram.")

Todos os testes passaram.


In [3]:
class NoUpdate:
    def update(self, w, grad):
        self.gradient = grad.copy()
        return w


mse = lambda Y, P: np.mean((Y - P) ** 2)
mse_derivative = lambda Y, P: 2 * (P - Y) / Y.size

np.random.seed(0)

X = np.random.rand(4, 3)
Y = np.random.rand(4, 2)

layer = DenseLayer(2, (3,)).initialize(NoUpdate())

# Forward e backward do método que implementaste
out = layer.forward_propagation(X, training=True)
output_error = mse_derivative(Y, out)
input_error = layer.backward_propagation(output_error)

analytic = layer.weights_optimizer.gradient.copy()

# Aproximar o gradiente alterando ligeiramente cada peso
numeric = np.zeros_like(layer.weights)
eps = 1e-6

for i in range(3):
    for j in range(2):
        original = layer.weights[i, j]

        layer.weights[i, j] = original + eps
        lp = mse(Y, layer.forward_propagation(X, training=True))

        layer.weights[i, j] = original - eps
        lm = mse(Y, layer.forward_propagation(X, training=True))

        layer.weights[i, j] = original
        numeric[i, j] = (lp - lm) / (2 * eps)

difference = np.abs(numeric - analytic).max()
print("Diferença máxima:", difference)

np.testing.assert_allclose(
    analytic, numeric, rtol=1e-5, atol=1e-7
)

print("Teste dos gradientes dos pesos passou.")

Diferença máxima: 2.154523781605633e-11
Teste dos gradientes dos pesos passou.


In [4]:
import numpy as np

from si.neural_networks.activation import (
    ReLUActivation,
    SigmoidActivation
)

x = np.array([-2.5, -1.2, -0.3, 0.4, 1.1, 2.7])
eps = 1e-6

for activation in [ReLUActivation(), SigmoidActivation()]:

    numeric = (
        activation.activation_function(x + eps)
        - activation.activation_function(x - eps)
    ) / (2 * eps)

    analytic = activation.derivative(x)

    difference = np.abs(numeric - analytic).max()

    print(activation.layer_name(), "— diferença máxima:", difference)

    np.testing.assert_allclose(
        analytic, numeric, rtol=1e-5, atol=1e-7
    )

print("Testes das derivadas passaram.")

ReLUActivation — diferença máxima: 1.397779669787269e-10
SigmoidActivation — diferença máxima: 6.56039597646263e-11
Testes das derivadas passaram.


In [5]:
import numpy as np

from si.neural_networks.layers import Dropout

np.random.seed(0)

X = np.random.rand(3, 4)

dropout = Dropout(probability=0.5, input_shape=(4,))

# Treino
output_train = dropout.forward_propagation(X, training=True)

print("Entrada:")
print(X)

print("\nMáscara:")
print(dropout.mask)

print("\nSaída durante o treino:")
print(output_train)

# Backward associado ao forward de treino
output_error = np.ones_like(X)
input_error = dropout.backward_propagation(output_error)

print("\nGradiente devolvido pelo backward:")
print(input_error)

# Avaliação
output_eval = dropout.forward_propagation(X, training=False)

print("\nSaída durante a avaliação:")
print(output_eval)

np.testing.assert_array_equal(output_eval, X)

print("\nTeste de avaliação passou.")

Entrada:
[[0.5488135  0.71518937 0.60276338 0.54488318]
 [0.4236548  0.64589411 0.43758721 0.891773  ]
 [0.96366276 0.38344152 0.79172504 0.52889492]]

Máscara:
[[1 1 0 0]
 [0 1 1 1]
 [1 1 0 1]]

Saída durante o treino:
[[1.09762701 1.43037873 0.         0.        ]
 [0.         1.29178823 0.87517442 1.783546  ]
 [1.92732552 0.76688304 0.         1.05778984]]

Gradiente devolvido pelo backward:
[[2. 2. 0. 0.]
 [0. 2. 2. 2.]
 [2. 2. 0. 2.]]

Saída durante a avaliação:
[[0.5488135  0.71518937 0.60276338 0.54488318]
 [0.4236548  0.64589411 0.43758721 0.891773  ]
 [0.96366276 0.38344152 0.79172504 0.52889492]]

Teste de avaliação passou.


In [6]:
import numpy as np

from si.neural_networks.activation import (
    TanhActivation,
    SoftmaxActivation
)

X = np.array([
    [2.0, 1.0, 0.1],
    [1.0, 2.0, 3.0]
])

output_error = np.array([
    [0.3, -0.4, 0.2],
    [-0.2, 0.7, 0.5]
])

eps = 1e-6

for activation in [TanhActivation(), SoftmaxActivation()]:

    output = activation.forward_propagation(X)
    analytic = activation.backward_propagation(output_error)

    numeric = np.zeros_like(X)

    # Alterar uma entrada de cada vez
    for index in np.ndindex(X.shape):
        X_plus = X.copy()
        X_minus = X.copy()

        X_plus[index] += eps
        X_minus[index] -= eps

        lp = np.sum(
            activation.activation_function(X_plus) * output_error
        )

        lm = np.sum(
            activation.activation_function(X_minus) * output_error
        )

        numeric[index] = (lp - lm) / (2 * eps)

    difference = np.abs(analytic - numeric).max()

    print(activation.layer_name(), "— diferença máxima:", difference)

    np.testing.assert_allclose(
        analytic, numeric, rtol=1e-5, atol=1e-7
    )

    if isinstance(activation, SoftmaxActivation):
        print("Probabilidades:")
        print(output)

        print("Soma de cada linha:", output.sum(axis=-1))

        np.testing.assert_allclose(
            output.sum(axis=-1), np.ones(X.shape[0])
        )

print("Testes de Tanh e Softmax passaram.")

TanhActivation — diferença máxima: 1.011104880377367e-10
SoftmaxActivation — diferença máxima: 6.21212248308467e-11
Probabilidades:
[[0.65900114 0.24243297 0.09856589]
 [0.09003057 0.24472847 0.66524096]]
Soma de cada linha: [1. 1.]
Testes de Tanh e Softmax passaram.
